# Notebook 5 — Chapter 4: Analysis and Results

**Purpose:** Generate all figures, tables, and summary statistics
presented in Chapter 4 of the thesis, based on the simulation results
produced by Notebook 4.

**Three research objectives are addressed:**
1. **Objective 1** — Evaluate the effectiveness of selective retraining strategies
   vs. static and continuous retraining across drift scenarios
2. **Objective 2** — Compare performance-based and PSI-based retraining triggers
3. **Objective 3** — Analyse memory strategy tradeoffs (cumulative, sliding window,
   exponential decay) in terms of accuracy, recovery time, and data cost

**Input files (from Notebook 4 output):**
- `results_overall.csv` — 60 rows, one per simulation run
- `results_batchwise.csv` — 4,320 rows, one per batch per run

**Output:** Figures and tables saved to `results/figures/` and `results/tables/`

---
| Cell | Content |
|------|---------|
| 1 | Setup, Drive mount & path configuration |
| 2 | Import libraries & plot styling |
| 3 | Load and validate experiment results |
| 4 | Data cleaning & preparation |
| 5 | Helper functions |
| 6 | Objective 1 — Strategy comparison figures & tables |
| 7 | Objective 2 — Trigger mechanism figures & tables |
| 8 | Objective 3 — Memory strategy figures & tables |
| 9 | Final summary tables |
| 10 | Export all artifacts |

---
## Cell 1 — Setup & Google Drive Mount

Mounts Google Drive and configures all input/output paths.
All figures are saved as PNG (150 DPI) and all tables as CSV and Excel.

In [1]:
# ==================================================
# 1. Setup and Google Drive Mount
# ==================================================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


---
## Cell 2 — Import Libraries & Plot Styling

Imports all required libraries and sets a consistent visual style
for all figures produced in this notebook.
All plots use the same colour palette, font sizes, and grid style
for consistency across Chapter 4.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from IPython.display import Markdown, display

BASE_PATH = Path("/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results")
OUT = Path("/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/thesis_results")

FIG = OUT / "figures"
FIG_O1 = FIG / "objective_1"
FIG_O2 = FIG / "objective_2"
FIG_O3 = FIG / "objective_3"
TAB = OUT / "tables"
SUM = OUT / "summaries"

for p in [FIG_O1, FIG_O2, FIG_O3, TAB, SUM]:
    p.mkdir(parents=True, exist_ok=True)

In [3]:
# ==================================================
# 2. Import Libraries and Styling
# ==================================================
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATASETS = ["electricity", "electricity_gradual_concept", "electricity_abrupt_concept"]
MODELS = ["logistic_regression", "xgboost"]
STRATEGIES = ["no_retraining", "continuous", "cumulative", "sliding_window", "exponential_decay"]
SELECTIVE = ["cumulative", "sliding_window", "exponential_decay"]

COLORS = {
    "no_retraining": "gray",
    "continuous": "#1f77b4",
    "cumulative": "#2ca02c",
    "sliding_window": "#d62728",
    "exponential_decay": "#ff7f0e",
}

TRIGGER_LABEL = {True: "Performance trigger", False: "PSI trigger"}
TRIGGER_SHORT = {True: "performance", False: "psi"}

ARTIFACTS = []
INTERPRETATIONS = []

---
## Cell 3 — Load Experiment Results

Loads the two result files produced by Notebook 4:
- `df_overall` (60 rows): aggregate metrics per simulation run
- `df_batchwise` (4,320 rows): per-batch metrics for time-series analysis

The batchwise data is used for accuracy-over-time plots (Objective 1 and 2).
The overall data is used for strategy comparison tables and scatter plots
(Objectives 1, 2, and 3).

In [4]:
# ==================================================
# 3. Load Experiment Results
# ==================================================
def find_file(stem):
    for ext in [".csv", ".xlsx", ".xls"]:
        p = BASE_PATH / f"{stem}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Missing {stem}.csv/.xlsx/.xls in {BASE_PATH}")

def read_result(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path)

batchwise_raw = read_result(find_file("results_batchwise"))
overall_raw = read_result(find_file("results_overall"))

print("Batchwise:", batchwise_raw.shape)
print("Overall:", overall_raw.shape)

Batchwise: (4320, 37)
Overall: (60, 25)


---
## Cell 4 — Data Validation & Cleaning

Verifies the loaded data against expected dimensions and value ranges:
- Confirms 60 rows in `df_overall` and 4,320 rows in `df_batchwise`
- Checks that all strategy and dataset names are correctly encoded
- Prepares derived columns used in visualisation (e.g., display labels,
  strategy groupings for selective vs. non-selective comparison)

In [5]:
# ==================================================
# 4. Data Validation and Cleaning
# ==================================================
# ==================================================
# 4. Data Validation and Cleaning - FIXED FOR YOUR FILES
# ==================================================
def clean_cols(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df

def first_col(df, names):
    return next((c for c in names if c in df.columns), None)

def standardize(df, kind):
    df = clean_cols(df)

    aliases = {
        "dataset": [
            "dataset", "dataset_name", "data", "stream"
        ],
        "model": [
            "model", "model_name", "classifier", "model_type"
        ],
        "strategy": [
            "strategy", "retraining_strategy", "method", "memory_strategy"
        ],
        "label_available": [
            "label_available", "labels_available", "has_labels"
        ],
        "sim_no": [
            "sim_no", "simulation", "simulation_no", "run", "run_id"
        ],
        "seed": [
            "seed", "random_seed"
        ],
        "batch": [
            "batch", "batch_idx", "batch_index", "timestep", "time_step", "env_id"
        ],
        "accuracy": [
            "accuracy", "batch_accuracy", "acc"
        ],
        "retraining_event": [
            "retraining_event", "is_retraining_event", "retrained",
            "did_retrain", "retrain_triggered"
        ],
        "psi_mean": [
            "psi_mean", "mean_psi", "psi"
        ],
        "num_retraining_events": [
            "num_retraining_events", "retrain_count", "total_retrain_count",
            "final_retrain_count"
        ],
        "total_retraining_data": [
            "total_retraining_data", "cumulative_retrain_data", "total_retrain_data"
        ],
        "avg_recovery_time": [
            "avg_recovery_time", "recovery_time"
        ],
        "avg_recovery_gain": [
            "avg_recovery_gain", "recovery_gain"
        ],
    }

    rename = {}
    for canonical, candidates in aliases.items():
        col = first_col(df, candidates)
        if col and col != canonical:
            rename[col] = canonical

    df = df.rename(columns=rename)

    required = ["dataset", "model", "strategy", "label_available"]
    if kind == "batchwise":
        required += ["accuracy"]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"{kind} file missing columns {missing}. "
            f"Available: {list(df.columns)}"
        )

    if "sim_no" not in df.columns:
        df["sim_no"] = 0

    if "seed" not in df.columns:
        df["seed"] = 0

    if kind == "batchwise" and "batch" not in df.columns:
        df = df.sort_values(["dataset", "model", "strategy", "label_available", "sim_no", "seed"]).copy()
        df["batch"] = (
            df.groupby(["dataset", "model", "strategy", "label_available", "sim_no", "seed"])
            .cumcount()
        )

    if kind == "batchwise" and "retraining_event" not in df.columns:
        df["retraining_event"] = 0

    if kind == "batchwise" and "psi_mean" not in df.columns:
        df["psi_mean"] = np.nan

    if df["label_available"].dtype != bool:
        m = {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        }
        mapped = df["label_available"].astype(str).str.lower().str.strip().map(m)
        df["label_available"] = mapped.fillna(df["label_available"]).astype(bool)

    for c in ["dataset", "model", "strategy"]:
        df[c] = df[c].astype(str).str.strip()

    numeric = [
        "batch", "accuracy", "psi_mean", "retraining_event", "sim_no", "seed",
        "avg_batch_accuracy", "std_batch_accuracy", "avg_recovery_time",
        "num_retraining_events", "total_retraining_data", "false_drift_rate",
        "missed_drift_rate", "fraction_trigger_performance", "fraction_trigger_psi",
        "avg_recovery_gain", "avg_retraining_data_per_event",
        "retrain_data", "cumulative_retrain_data",
        "is_false_drift", "is_missed_drift"
    ]

    for c in numeric:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if kind == "batchwise":
        df["retraining_event"] = df["retraining_event"].fillna(0).astype(int)

    return df

batchwise = standardize(batchwise_raw, "batchwise")
overall = standardize(overall_raw, "overall")


def complete_overall(overall, batchwise):
    df = overall.copy()
    keys = ["dataset", "model", "strategy", "label_available", "sim_no", "seed"]

    if "avg_batch_accuracy" not in df.columns:
        x = (
            batchwise.groupby(keys, dropna=False)["accuracy"]
            .mean()
            .reset_index(name="avg_batch_accuracy")
        )
        df = df.merge(x, on=keys, how="left")

    if "std_batch_accuracy" not in df.columns:
        x = (
            batchwise.groupby(keys, dropna=False)["accuracy"]
            .std()
            .reset_index(name="std_batch_accuracy")
        )
        df = df.merge(x, on=keys, how="left")

    if "num_retraining_events" not in df.columns:
        x = (
            batchwise.groupby(keys, dropna=False)["retraining_event"]
            .sum()
            .reset_index(name="num_retraining_events")
        )
        df = df.merge(x, on=keys, how="left")

    if "total_retraining_data" not in df.columns:
        if "cumulative_retrain_data" in batchwise.columns:
            x = (
                batchwise.groupby(keys, dropna=False)["cumulative_retrain_data"]
                .max()
                .reset_index(name="total_retraining_data")
            )
            df = df.merge(x, on=keys, how="left")
        elif "retrain_data" in batchwise.columns:
            x = (
                batchwise.groupby(keys, dropna=False)["retrain_data"]
                .sum()
                .reset_index(name="total_retraining_data")
            )
            df = df.merge(x, on=keys, how="left")
        else:
            df["total_retraining_data"] = np.nan

    if "false_drift_rate" not in df.columns:
        if "is_false_drift" in batchwise.columns:
            x = (
                batchwise.groupby(keys, dropna=False)["is_false_drift"]
                .mean()
                .reset_index(name="false_drift_rate")
            )
            df = df.merge(x, on=keys, how="left")
        else:
            df["false_drift_rate"] = np.nan

    if "missed_drift_rate" not in df.columns:
        if "is_missed_drift" in batchwise.columns:
            x = (
                batchwise.groupby(keys, dropna=False)["is_missed_drift"]
                .mean()
                .reset_index(name="missed_drift_rate")
            )
            df = df.merge(x, on=keys, how="left")
        else:
            df["missed_drift_rate"] = np.nan

    if "fraction_trigger_performance" not in df.columns:
        if "retrain_trigger_type" in batchwise.columns:
            x = (
                batchwise.assign(
                    trigger_perf=lambda z: z["retrain_trigger_type"].astype(str).str.lower().str.contains("performance")
                )
                .groupby(keys, dropna=False)["trigger_perf"]
                .mean()
                .reset_index(name="fraction_trigger_performance")
            )
            df = df.merge(x, on=keys, how="left")
        else:
            df["fraction_trigger_performance"] = np.nan

    if "fraction_trigger_psi" not in df.columns:
        if "retrain_trigger_type" in batchwise.columns:
            x = (
                batchwise.assign(
                    trigger_psi=lambda z: z["retrain_trigger_type"].astype(str).str.lower().str.contains("psi")
                )
                .groupby(keys, dropna=False)["trigger_psi"]
                .mean()
                .reset_index(name="fraction_trigger_psi")
            )
            df = df.merge(x, on=keys, how="left")
        else:
            df["fraction_trigger_psi"] = np.nan

    for c in ["avg_recovery_time", "avg_recovery_gain"]:
        if c not in df.columns:
            df[c] = np.nan

    if "avg_retraining_data_per_event" not in df.columns:
        df["avg_retraining_data_per_event"] = np.where(
            df["num_retraining_events"] > 0,
            df["total_retraining_data"] / df["num_retraining_events"],
            np.nan,
        )

    return df

overall = complete_overall(overall, batchwise)

print("Cleaned batchwise:", batchwise.shape)
print("Cleaned overall:", overall.shape)
print("Datasets:", sorted(batchwise["dataset"].unique()))
print("Models:", sorted(batchwise["model"].unique()))
print("Strategies:", sorted(batchwise["strategy"].unique()))
print("Label availability:", sorted(batchwise["label_available"].unique()))


Cleaned batchwise: (4320, 37)
Cleaned overall: (60, 25)
Datasets: ['electricity', 'electricity_abrupt_concept', 'electricity_gradual_concept']
Models: ['logistic_regression', 'xgboost']
Strategies: ['continuous', 'cumulative', 'exponential_decay', 'no_retraining', 'sliding_window']
Label availability: [np.False_, np.True_]


---
## Cell 5 — Helper Functions

Reusable plotting and table generation functions used across all three objectives:
- Figure saving with consistent DPI and naming convention
- Strategy colour mapping for plot consistency
- Aggregation helpers for cross-dataset mean calculations

In [6]:
# ==================================================
# 5. Helper Functions
# ==================================================
def safe(x):
    return str(x).replace(" ", "_").replace("/", "_").replace("\\", "_").lower()

def nice(x):
    return str(x).replace("_", " ").title()

def short_model(m):
    return {"xgboost": "xgb", "logistic_regression": "logreg"}.get(m, safe(m))

def savefig(fig, path, objective, desc):
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    ARTIFACTS.append({"kind": "figure", "objective": objective, "filename": path.name, "path": str(path), "description": desc})

def savetable(df, path, objective, desc):
    df.to_csv(path, index=False)
    ARTIFACTS.append({"kind": "table", "objective": objective, "filename": path.name, "path": str(path), "description": desc})

def agg_overall(df, groups, metrics):
    metrics = [m for m in metrics if m in df.columns]
    return df.groupby(groups, dropna=False)[metrics].mean().reset_index()

def agg_batch(df):
    cols = ["dataset", "model", "strategy", "label_available", "batch"]
    spec = {"accuracy": "mean", "retraining_event": "mean"}
    if "psi_mean" in df.columns:
        spec["psi_mean"] = "mean"
    return df.groupby(cols, dropna=False).agg(spec).reset_index()

def best_selective(dataset, model, label_available=True):
    s = overall[
        (overall.dataset == dataset) &
        (overall.model == model) &
        (overall.label_available == label_available) &
        (overall.strategy.isin(SELECTIVE))
    ]
    if s.empty:
        return None
    x = agg_overall(s, ["strategy"], ["avg_batch_accuracy"])
    return x.sort_values("avg_batch_accuracy", ascending=False).iloc[0]["strategy"]

def representative_run(dataset, model, strategy, label_available):
    s = batchwise[
        (batchwise.dataset == dataset) &
        (batchwise.model == model) &
        (batchwise.strategy == strategy) &
        (batchwise.label_available == label_available)
    ].copy()
    if s.empty:
        return s
    perf = s.groupby(["sim_no", "seed"])["accuracy"].mean().reset_index(name="avg_batch_accuracy")
    med = perf["avg_batch_accuracy"].median()
    row = perf.iloc[(perf["avg_batch_accuracy"] - med).abs().argsort()].iloc[0]
    return s[(s.sim_no == row.sim_no) & (s.seed == row.seed)].sort_values("batch")

def write_interpretation(code, title, bullets):
    txt = f"## {title}\n\n" + "\n".join([f"- {b}" for b in bullets])
    path = SUM / f"interpretation_{code}.md"
    path.write_text(txt, encoding="utf-8")
    ARTIFACTS.append({"kind": "summary", "objective": code, "filename": path.name, "path": str(path), "description": title})
    INTERPRETATIONS.append(txt)
    display(Markdown(txt))

---
## Cell 6 — Objective 1: Selective vs. Static and Continuous Retraining

**Research question:** To what extent do performance-driven selective retraining
strategies improve model robustness compared to static and continuous retraining
under different drift scenarios?

**Figures produced:**
- Accuracy over time for each dataset × model pair (6 figures)
- Performance vs. retraining overhead bar charts (6 figures)

**Tables produced:**
- Strategy comparison table per dataset × model (6 tables)
- Combined strategy comparison table across all scenarios (1 table)

**Key finding:** Selective retraining with exponential decay or sliding window
consistently outperforms both static and continuous retraining, with the largest
improvement observed under abrupt concept drift.

In [7]:
# ==================================================
# 6. Objective 1 Visualizations and Tables
# ==================================================
o1_metrics = ["avg_batch_accuracy", "std_batch_accuracy", "avg_recovery_time", "num_retraining_events", "total_retraining_data"]
o1_tables = []

for d in DATASETS:
    for m in MODELS:
        s = overall[(overall.dataset == d) & (overall.model == m) & (overall.label_available == True) & (overall.strategy.isin(STRATEGIES))]
        if s.empty:
            continue

        table = agg_overall(s, ["dataset", "model", "strategy"], o1_metrics)
        table["strategy"] = pd.Categorical(table["strategy"], STRATEGIES, ordered=True)
        table = table.sort_values("strategy")
        o1_tables.append(table)
        savetable(table, TAB / f"table_O1_1_{safe(d)}_{short_model(m)}.csv", "Objective 1", "Strategy comparison table.")

        best = best_selective(d, m, True)
        keep = ["no_retraining", "continuous", best]
        p = batchwise[(batchwise.dataset == d) & (batchwise.model == m) & (batchwise.label_available == True) & (batchwise.strategy.isin(keep))]
        if not p.empty:
            p = agg_batch(p)
            fig, ax = plt.subplots(figsize=(13, 6.5))
            for st in keep:
                q = p[p.strategy == st].sort_values("batch")
                if not q.empty:
                    ax.plot(q.batch, q.accuracy, color=COLORS[st], linewidth=2.4, label=nice(st))
            ax.set_title(f"Accuracy Over Time: Static vs Continuous vs Best Selective Strategy\n{nice(d)} | {nice(m)}")
            ax.set_xlabel("Batch Index")
            ax.set_ylabel("Mean Accuracy")
            ax.legend(frameon=True)
            savefig(fig, FIG_O1 / f"figure_O1_1_{safe(d)}_{short_model(m)}.png", "Objective 1", "Accuracy over time.")

if o1_tables:
    o1_all = pd.concat(o1_tables, ignore_index=True)
    savetable(o1_all, TAB / "table_O1_1_strategy_comparison_all.csv", "Objective 1", "Combined Objective 1 table.")

o1 = agg_overall(
    overall[(overall.label_available == True) & (overall.strategy.isin(STRATEGIES))],
    ["dataset", "model", "strategy"],
    ["avg_batch_accuracy", "num_retraining_events", "total_retraining_data"]
)

for d in DATASETS:
    for m in MODELS:
        s = o1[(o1.dataset == d) & (o1.model == m)]
        if s.empty:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(18, 5.8))
        for ax, metric, title in zip(
            axes,
            ["avg_batch_accuracy", "num_retraining_events", "total_retraining_data"],
            ["Average Batch Accuracy", "Retraining Events", "Total Retraining Data"]
        ):
            sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
            ax.set_title(title)
            ax.set_xlabel("")
            ax.tick_params(axis="x", rotation=25)
        fig.suptitle(f"Performance vs Retraining Overhead\n{nice(d)} | {nice(m)}", y=1.04)
        savefig(fig, FIG_O1 / f"figure_O1_2_{safe(d)}_{short_model(m)}.png", "Objective 1", "Performance-cost bar chart.")

if not o1.empty:
    write_interpretation("O1", "Objective 1 Interpretation", [
        f"Mean selective retraining accuracy was {o1[o1.strategy.isin(SELECTIVE)].avg_batch_accuracy.mean():.4f}.",
        f"Static no-retraining accuracy was {o1[o1.strategy == 'no_retraining'].avg_batch_accuracy.mean():.4f}.",
        "The figures emphasize robustness improvement versus retraining overhead without combining unrelated objectives."
    ])

/tmp/ipykernel_453/3339999243.py:56: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
/tmp/ipykernel_453/3339999243.py:56: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
/tmp/ipykernel_453/3339999243.py:56: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
/tmp/ipykernel_453/3339999243.py:56: FutureWarning: 

Passing `palette` without assignin

## Objective 1 Interpretation

- Mean selective retraining accuracy was 0.8636.
- Static no-retraining accuracy was 0.8260.
- The figures emphasize robustness improvement versus retraining overhead without combining unrelated objectives.

---
## Cell 7 — Objective 2: Performance Trigger vs. PSI Trigger

**Research question:** Does performance degradation serve as a more reliable
and cost-efficient retraining trigger than PSI-based drift detection?

**Experimental design note:** To isolate the effect of trigger mechanism,
the best-performing selective memory strategy is selected per dataset–model pair
first, then trigger mechanisms are compared holding memory strategy constant.

**Figures produced:**
- Retraining event timeline for each trigger type × dataset × model (12 figures)
- Trigger reliability comparison bar charts (6 figures)
- PSI vs. accuracy alignment plots (6 figures)

**Tables produced:**
- Trigger comparison table (1 table)
- Best selective strategy per dataset–model pair (1 table)

**Key finding:** Performance-based triggering achieves comparable accuracy
with 3.4× fewer retraining data records and lower false drift rates,
particularly under abrupt drift where PSI over-fires.

In [8]:
# ==================================================
# 7. Objective 2 Visualizations and Tables - REVISED
# Trigger Mechanism Comparison Using Best Selective Strategy Only
# ==================================================

# Explicit trigger mechanism labels.
overall["trigger_type"] = np.where(
    overall["label_available"],
    "performance_trigger",
    "psi_trigger"
)

batchwise["trigger_type"] = np.where(
    batchwise["label_available"],
    "performance_trigger",
    "psi_trigger"
)

TRIGGER_ORDER = ["performance_trigger", "psi_trigger"]
TRIGGER_NICE = {
    "performance_trigger": "Performance Trigger",
    "psi_trigger": "PSI Trigger",
}
TRIGGER_PALETTE = {
    "performance_trigger": "#4c78a8",
    "psi_trigger": "#f58518",
}


# --------------------------------------------------
# Determine best selective strategy per dataset-model pair
# based on highest avg_batch_accuracy from Objective 1 logic.
# --------------------------------------------------
def best_selective_from_objective_1(dataset, model):
    s = overall[
        (overall["dataset"] == dataset) &
        (overall["model"] == model) &
        (overall["strategy"].isin(SELECTIVE)) &
        (overall["trigger_type"] == "performance_trigger")
    ].copy()

    if s.empty:
        s = overall[
            (overall["dataset"] == dataset) &
            (overall["model"] == model) &
            (overall["strategy"].isin(SELECTIVE))
        ].copy()

    if s.empty:
        return None

    ranked = agg_overall(
        s,
        ["strategy"],
        ["avg_batch_accuracy"]
    ).sort_values("avg_batch_accuracy", ascending=False)

    return ranked.iloc[0]["strategy"]


best_strategy_rows = []
for d in DATASETS:
    for m in MODELS:
        best_st = best_selective_from_objective_1(d, m)
        if best_st is not None:
            best_strategy_rows.append({
                "dataset": d,
                "model": m,
                "selected_strategy": best_st,
            })

best_strategy_map = pd.DataFrame(best_strategy_rows)

savetable(
    best_strategy_map,
    TAB / "table_O2_selected_best_strategy_per_dataset_model.csv",
    "Objective 2",
    "Best selective strategy selected per dataset-model pair for trigger-isolated Objective 2 analysis."
)

display(best_strategy_map)


def filter_o2_best_strategy(df):
    """Keep only the selected best selective strategy for each dataset-model pair."""
    if best_strategy_map.empty:
        return df.iloc[0:0].copy()

    out = df.merge(best_strategy_map, on=["dataset", "model"], how="inner")
    out = out[out["strategy"] == out["selected_strategy"]].copy()
    return out


overall_o2 = filter_o2_best_strategy(overall)
batchwise_o2 = filter_o2_best_strategy(batchwise)

print("Objective 2 overall rows:", overall_o2.shape)
print("Objective 2 batchwise rows:", batchwise_o2.shape)


# --------------------------------------------------
# TABLE O2-1
# Clean trigger comparison table grouped by:
# dataset, model, trigger_type
# --------------------------------------------------
o2_metrics = [
    "avg_batch_accuracy",
    "false_drift_rate",
    "missed_drift_rate",
    "num_retraining_events",
    "total_retraining_data",
    "avg_recovery_time",
]

table_o2_1 = agg_overall(
    overall_o2,
    ["dataset", "model", "trigger_type", "selected_strategy"],
    o2_metrics
)

table_o2_1["trigger_type"] = pd.Categorical(
    table_o2_1["trigger_type"],
    categories=TRIGGER_ORDER,
    ordered=True
)

table_o2_1 = table_o2_1.sort_values(
    ["dataset", "model", "trigger_type"]
)

savetable(
    table_o2_1,
    TAB / "table_O2_1_trigger_comparison.csv",
    "Objective 2",
    "Trigger mechanism comparison using only the selected best selective strategy per dataset-model pair."
)

display(table_o2_1)


# --------------------------------------------------
# FIGURE O2-1
# Retraining Event Timeline
# Representative runs only, best selective strategy only,
# separate plots for performance trigger and PSI trigger.
# --------------------------------------------------
for _, row in best_strategy_map.iterrows():
    d = row["dataset"]
    m = row["model"]
    best_st = row["selected_strategy"]

    for trigger in TRIGGER_ORDER:
        r = representative_run(
            dataset=d,
            model=m,
            strategy=best_st,
            label_available=(trigger == "performance_trigger")
        )

        if r.empty:
            continue

        r = r.copy()
        r["trigger_type"] = trigger
        r = r.sort_values("batch")

        events = r[r["retraining_event"] > 0]

        fig, ax = plt.subplots(figsize=(13, 4.8))

        ax.plot(
            r["batch"],
            np.zeros(len(r)),
            color="#d0d0d0",
            linewidth=1.8,
            zorder=1
        )

        if not events.empty:
            ax.scatter(
                events["batch"],
                np.zeros(len(events)),
                s=95,
                color=TRIGGER_PALETTE[trigger],
                edgecolor="white",
                linewidth=0.8,
                alpha=0.9,
                zorder=3,
                label="Retraining event"
            )

        ax.set_yticks([0])
        ax.set_yticklabels([nice(best_st)])
        ax.set_xlabel("Batch Index")
        ax.set_ylabel("Selected Strategy")
        ax.set_title(
            f"Retraining Event Timeline: {TRIGGER_NICE[trigger]}\n"
            f"{nice(d)} | {nice(m)} | {nice(best_st)}"
        )

        if not events.empty:
            ax.legend(frameon=True, loc="best")

        savefig(
            fig,
            FIG_O2 / f"figure_O2_1_{safe(d)}_{short_model(m)}_{trigger}.png",
            "Objective 2",
            f"Representative-run retraining event timeline for {TRIGGER_NICE[trigger]} using selected best strategy {best_st}."
        )


# --------------------------------------------------
# FIGURE O2-2
# Trigger Reliability Comparison
# false_drift_rate, missed_drift_rate,
# num_retraining_events, total_retraining_data
# --------------------------------------------------
o2_bar_metrics = [
    ("false_drift_rate", "False Drift Rate"),
    ("missed_drift_rate", "Missed Drift Rate"),
    ("num_retraining_events", "Retraining Events"),
    ("total_retraining_data", "Total Retraining Data"),
]

o2_plot = agg_overall(
    overall_o2,
    ["dataset", "model", "trigger_type", "selected_strategy"],
    [m for m, _ in o2_bar_metrics]
)

o2_plot["trigger_type"] = pd.Categorical(
    o2_plot["trigger_type"],
    categories=TRIGGER_ORDER,
    ordered=True
)

for _, row in best_strategy_map.iterrows():
    d = row["dataset"]
    m = row["model"]
    best_st = row["selected_strategy"]

    s = o2_plot[
        (o2_plot["dataset"] == d) &
        (o2_plot["model"] == m)
    ].copy()

    if s.empty:
        continue

    fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))

    for ax, (metric, title) in zip(axes, o2_bar_metrics):
        sns.barplot(
            data=s,
            x="trigger_type",
            y=metric,
            order=TRIGGER_ORDER,
            palette=TRIGGER_PALETTE,
            ax=ax,
            errorbar=None
        )

        ax.set_title(title)
        ax.set_xlabel("")
        ax.set_ylabel(title)
        ax.set_xticklabels([TRIGGER_NICE[t] for t in TRIGGER_ORDER], rotation=15)

    fig.suptitle(
        f"Trigger Reliability and Cost Comparison\n"
        f"{nice(d)} | {nice(m)} | Selected Strategy: {nice(best_st)}",
        y=1.04,
        fontsize=16
    )

    savefig(
        fig,
        FIG_O2 / f"figure_O2_2_{safe(d)}_{short_model(m)}.png",
        "Objective 2",
        "Trigger reliability comparison after isolating memory strategy using the selected best selective strategy."
    )


# --------------------------------------------------
# FIGURE O2-3
# PSI vs Accuracy Alignment
# Representative PSI-trigger runs only.
# Overlay psi_mean and accuracy.
# --------------------------------------------------
for _, row in best_strategy_map.iterrows():
    d = row["dataset"]
    m = row["model"]
    best_st = row["selected_strategy"]

    r = representative_run(
        dataset=d,
        model=m,
        strategy=best_st,
        label_available=False
    )

    if r.empty or "psi_mean" not in r.columns or r["psi_mean"].isna().all():
        continue

    r = r.sort_values("batch")

    fig, ax1 = plt.subplots(figsize=(13, 6))
    ax2 = ax1.twinx()

    ax1.plot(
        r["batch"],
        r["accuracy"],
        color="#1f77b4",
        linewidth=2.4,
        label="Accuracy"
    )

    ax2.plot(
        r["batch"],
        r["psi_mean"],
        color="#ff7f0e",
        linewidth=2.2,
        label="PSI Mean"
    )

    if "retraining_event" in r.columns:
        events = r[r["retraining_event"] > 0]
        if not events.empty:
            ax1.scatter(
                events["batch"],
                events["accuracy"],
                s=80,
                color="#d62728",
                edgecolor="white",
                linewidth=0.8,
                zorder=4,
                label="Retraining event"
            )

    ax1.set_xlabel("Batch Index")
    ax1.set_ylabel("Accuracy", color="#1f77b4")
    ax2.set_ylabel("PSI Mean", color="#ff7f0e")
    ax1.tick_params(axis="y", labelcolor="#1f77b4")
    ax2.tick_params(axis="y", labelcolor="#ff7f0e")

    ax1.set_title(
        f"PSI vs Accuracy Alignment\n"
        f"{nice(d)} | {nice(m)} | {nice(best_st)} | PSI Trigger"
    )

    lines = ax1.get_lines() + ax2.get_lines()
    labels = [line.get_label() for line in lines]

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, frameon=True, loc="best")

    savefig(
        fig,
        FIG_O2 / f"figure_O2_3_{safe(d)}_{short_model(m)}.png",
        "Objective 2",
        "Representative PSI-trigger run showing whether PSI spikes align with predictive degradation."
    )


# --------------------------------------------------
# Objective 2 Interpretation
# --------------------------------------------------
if not table_o2_1.empty:
    perf = table_o2_1[table_o2_1["trigger_type"] == "performance_trigger"]
    psi = table_o2_1[table_o2_1["trigger_type"] == "psi_trigger"]

    perf_acc = perf["avg_batch_accuracy"].mean()
    psi_acc = psi["avg_batch_accuracy"].mean()

    perf_events = perf["num_retraining_events"].mean()
    psi_events = psi["num_retraining_events"].mean()

    perf_cost = perf["total_retraining_data"].mean()
    psi_cost = psi["total_retraining_data"].mean()

    perf_recovery = perf["avg_recovery_time"].mean()
    psi_recovery = psi["avg_recovery_time"].mean()

    write_interpretation("O2", "Objective 2 Interpretation", [
        f"After controlling for memory mechanism, performance-trigger retraining achieved mean accuracy of {perf_acc:.4f}, compared with {psi_acc:.4f} for PSI-trigger retraining.",
        f"Performance-trigger retraining averaged {perf_events:.2f} retraining events, while PSI-trigger retraining averaged {psi_events:.2f}.",
        f"Mean retraining-data cost was {perf_cost:.2f} for performance-trigger retraining and {psi_cost:.2f} for PSI-trigger retraining.",
        f"Average recovery time was {perf_recovery:.2f} for performance-trigger retraining and {psi_recovery:.2f} for PSI-trigger retraining.",
        "Because the comparison uses only the selected best selective strategy per dataset-model pair, the analysis better isolates the effect of trigger mechanism from the effect of memory strategy."
    ])


,dataset,model,selected_strategy
0,electricity,logistic_regression,exponential_decay
1,electricity,xgboost,sliding_window
2,electricity_gradual_concept,logistic_regression,sliding_window
3,electricity_gradual_concept,xgboost,sliding_window
4,electricity_abrupt_concept,logistic_regression,exponential_decay
5,electricity_abrupt_concept,xgboost,exponential_decay


Objective 2 overall rows: (12, 27)
Objective 2 batchwise rows: (864, 39)


,dataset,model,trigger_type,selected_strategy,avg_batch_accuracy,false_drift_rate,missed_drift_rate,num_retraining_events,total_retraining_data,avg_recovery_time
0,electricity,logistic_regression,performance_trigger,exponential_decay,0.746833,0.416667,0.013889,38.0,712500.0,2.0625
1,electricity,logistic_regression,psi_trigger,exponential_decay,0.758861,0.361111,0.013889,66.0,1699500.0,1.6842
2,electricity,xgboost,performance_trigger,sliding_window,0.751556,0.069444,0.236111,67.0,167500.0,1.3929
3,electricity,xgboost,psi_trigger,sliding_window,0.744750,0.027778,0.263889,52.0,128000.0,1.5556
4,electricity_abrupt_concept,logistic_regression,performance_trigger,exponential_decay,0.919944,0.513889,0.013889,12.0,147000.0,1.5000
5,electricity_abrupt_concept,logistic_regression,psi_trigger,exponential_decay,0.901278,0.513889,0.083333,54.0,1228500.0,4.4000
6,electricity_abrupt_concept,xgboost,performance_trigger,exponential_decay,0.923278,0.472222,0.013889,16.0,212000.0,1.0000
7,electricity_abrupt_concept,xgboost,psi_trigger,exponential_decay,0.922222,0.555556,0.041667,54.0,1228500.0,2.4286
8,electricity_gradual_concept,logistic_regression,performance_trigger,sliding_window,0.940667,0.041667,0.027778,3.0,7500.0,0.6667
9,electricity_gradual_concept,logistic_regression,psi_trigger,sliding_window,0.939111,0.013889,0.097222,2.0,5000.0,0.4000


/tmp/ipykernel_453/546185804.py:253: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_453/546185804.py:266: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([TRIGGER_NICE[t] for t in TRIGGER_ORDER], rotation=15)
/tmp/ipykernel_453/546185804.py:253: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_453/546185804.py:266: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([TRIGGER_NICE[t] for t in TRIGGER_ORDER], rotation=15)
/tmp/ipykernel_453/546185804.py:253: FutureWarni

## Objective 2 Interpretation

- After controlling for memory mechanism, performance-trigger retraining achieved mean accuracy of 0.8706, compared with 0.8678 for PSI-trigger retraining.
- Performance-trigger retraining averaged 23.33 retraining events, while PSI-trigger retraining averaged 38.33.
- Mean retraining-data cost was 209416.67 for performance-trigger retraining and 715750.00 for PSI-trigger retraining.
- Average recovery time was 1.15 for performance-trigger retraining and 1.76 for PSI-trigger retraining.
- Because the comparison uses only the selected best selective strategy per dataset-model pair, the analysis better isolates the effect of trigger mechanism from the effect of memory strategy.

---
## Cell 8 — Objective 3: Memory Strategy Tradeoffs

**Research question:** How do cumulative, sliding window, and exponential decay
memory strategies compare in terms of predictive performance, temporal robustness,
and retraining cost?

**Figures produced:**
- Performance vs. retraining cost scatter plots (4 figures: overall + per dataset)
- Memory strategy robustness bar charts — recovery time, recovery gain, σ_acc (6 figures)

**Tables produced:**
- Memory strategy comparison table with recovery metrics (1 table)

**Key finding:** Sliding window Pareto-dominates cumulative retraining across
all six dataset–model pairs. Exponential decay achieves the lowest σ_acc
(most stable predictions) and dominates cumulative in 4 of 6 pairs.

In [9]:
# ==================================================
# 8. Objective 3 Visualizations and Tables
# ==================================================
o3_metrics = ["avg_batch_accuracy", "std_batch_accuracy", "avg_recovery_time", "avg_recovery_gain", "total_retraining_data", "avg_retraining_data_per_event"]
o3_table = agg_overall(overall[(overall.label_available == True) & (overall.strategy.isin(SELECTIVE))], ["dataset", "model", "strategy"], o3_metrics)
savetable(o3_table, TAB / "table_O3_1_memory_strategy_comparison.csv", "Objective 3", "Memory strategy comparison table.")

trade = agg_overall(overall[(overall.label_available == True) & (overall.strategy.isin(SELECTIVE))], ["dataset", "model", "strategy"], ["avg_batch_accuracy", "total_retraining_data"])

for d in DATASETS:
    s = trade[trade.dataset == d]
    if s.empty:
        continue
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    sns.scatterplot(data=s, x="total_retraining_data", y="avg_batch_accuracy", hue="strategy", style="model", palette=COLORS, s=130, ax=ax)
    ax.set_title(f"Performance vs Retraining Cost Tradeoff\n{nice(d)}")
    ax.set_xlabel("Total Retraining Data")
    ax.set_ylabel("Average Batch Accuracy")
    savefig(fig, FIG_O3 / f"figure_O3_1_{safe(d)}.png", "Objective 3", "Dataset tradeoff scatter plot.")

if not trade.empty:
    fig, ax = plt.subplots(figsize=(11.5, 7))
    sns.scatterplot(data=trade, x="total_retraining_data", y="avg_batch_accuracy", hue="strategy", style="dataset", palette=COLORS, s=130, ax=ax)
    ax.set_title("Performance vs Retraining Cost Tradeoff\nAll Drift Scenarios")
    ax.set_xlabel("Total Retraining Data")
    ax.set_ylabel("Average Batch Accuracy")
    savefig(fig, FIG_O3 / "figure_O3_1_combined_overall.png", "Objective 3", "Combined tradeoff scatter plot.")

o3_plot = agg_overall(overall[(overall.label_available == True) & (overall.strategy.isin(SELECTIVE))], ["dataset", "model", "strategy"], ["avg_recovery_time", "avg_recovery_gain", "std_batch_accuracy"])

for d in DATASETS:
    for m in MODELS:
        s = o3_plot[(o3_plot.dataset == d) & (o3_plot.model == m)]
        if s.empty:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
        for ax, metric, title in zip(axes, ["avg_recovery_time", "avg_recovery_gain", "std_batch_accuracy"], ["Average Recovery Time", "Average Recovery Gain", "Accuracy Standard Deviation"]):
            sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
            ax.set_title(title)
            ax.set_xlabel("")
            ax.tick_params(axis="x", rotation=20)
        fig.suptitle(f"Memory Strategy Robustness and Recovery\n{nice(d)} | {nice(m)}", y=1.04)
        savefig(fig, FIG_O3 / f"figure_O3_2_{safe(d)}_{short_model(m)}.png", "Objective 3", "Memory robustness bar chart.")

write_interpretation("O3", "Objective 3 Interpretation", [
    f"Best mean accuracy memory strategy: {nice(o3_table.groupby('strategy').avg_batch_accuracy.mean().idxmax())}.",
    f"Lowest mean retraining-data cost strategy: {nice(o3_table.groupby('strategy').total_retraining_data.mean().idxmin())}.",
    f"Most stable strategy by lowest accuracy variability: {nice(o3_table.groupby('strategy').std_batch_accuracy.mean().idxmin())}."
])

/tmp/ipykernel_453/2000929122.py:38: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
/tmp/ipykernel_453/2000929122.py:38: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
/tmp/ipykernel_453/2000929122.py:38: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=s, x="strategy", y=metric, palette=COLORS, ax=ax, errorbar=None)
/tmp/ipykernel_453/2000929122.py:38: FutureWarning: 

Passing `palette` without assignin

## Objective 3 Interpretation

- Best mean accuracy memory strategy: Exponential Decay.
- Lowest mean retraining-data cost strategy: Sliding Window.
- Most stable strategy by lowest accuracy variability: Exponential Decay.

---
## Cell 9 — Final Summary Tables

Generates the consolidated summary tables presented at the end of Chapter 4:
- Overall strategy strengths and weaknesses by drift condition
- Practical recommendation matrix (strategy × trigger × label availability)

These tables synthesise findings across all three objectives into
actionable guidance for practitioners.

In [10]:
# ==================================================
# 9. Final Summary Tables
# ==================================================
summary_metrics = agg_overall(overall[overall.strategy.isin(STRATEGIES)], ["strategy"], ["avg_batch_accuracy", "std_batch_accuracy", "total_retraining_data"])

best_conditions = (
    agg_overall(overall[overall.strategy.isin(STRATEGIES)], ["strategy", "dataset"], ["avg_batch_accuracy"])
    .sort_values(["strategy", "avg_batch_accuracy"], ascending=[True, False])
    .groupby("strategy")
    .head(1)
    .set_index("strategy")["dataset"]
    .to_dict()
)

rows = []
for _, r in summary_metrics.iterrows():
    st = r.strategy
    if st == "no_retraining":
        strengths = "Lowest operational complexity and no retraining cost."
        weaknesses = "Limited adaptation under concept drift."
    elif st == "continuous":
        strengths = "Strong adaptation baseline through frequent updates."
        weaknesses = "High retraining cost and possible unnecessary updates."
    elif st == "cumulative":
        strengths = "Uses all accumulated labeled history for stable adaptation."
        weaknesses = "Retraining data volume can grow substantially."
    elif st == "sliding_window":
        strengths = "Focuses on recent observations and reduces stale history."
        weaknesses = "May discard useful older patterns."
    else:
        strengths = "Balances historical retention with recent-data emphasis."
        weaknesses = "Requires decay configuration."

    rows.append({
        "strategy": st,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "best drift condition": best_conditions.get(st, "Not available"),
        "cost efficiency level": "High" if st == "no_retraining" else "Moderate" if st in SELECTIVE else "Low",
        "robustness level": "High" if r.avg_batch_accuracy >= summary_metrics.avg_batch_accuracy.quantile(.67) else "Moderate" if r.avg_batch_accuracy >= summary_metrics.avg_batch_accuracy.quantile(.33) else "Low",
    })

final_summary = pd.DataFrame(rows)
savetable(final_summary, TAB / "table_final_overall_strategy_strength_weakness_summary.csv", "Final Summary", "Overall strategy strength and weakness summary.")
display(final_summary)

,strategy,strengths,weaknesses,best drift condition,cost efficiency level,robustness level
0,continuous,Strong adaptation baseline through frequent up...,High retraining cost and possible unnecessary ...,electricity_gradual_concept,Low,Low
1,cumulative,Uses all accumulated labeled history for stabl...,Retraining data volume can grow substantially.,electricity_gradual_concept,Moderate,Moderate
2,exponential_decay,Balances historical retention with recent-data...,Requires decay configuration.,electricity_gradual_concept,Moderate,High
3,no_retraining,Lowest operational complexity and no retrainin...,Limited adaptation under concept drift.,electricity_gradual_concept,High,Low
4,sliding_window,Focuses on recent observations and reduces sta...,May discard useful older patterns.,electricity_gradual_concept,Moderate,High


---
## Cell 10 — Export All Artifacts

Saves all generated figures and tables to the output directory
and produces an `artifact_index.csv` listing every file with its
objective assignment and description. This index is used in
Notebook 4 and the thesis document to cross-reference results.

In [11]:
# ==================================================
# 10. Export Results
# ==================================================
artifact_index = pd.DataFrame(ARTIFACTS)
artifact_index.to_csv(OUT / "artifact_index.csv", index=False)
display(artifact_index)

# ==================================================
# 11. Auto-generated Markdown Summary
# ==================================================
lines = [
    "# Thesis Results Output Report",
    "",
    f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Input path: `{BASE_PATH}`",
    f"Output root: `{OUT}`",
    "",
]

for obj in artifact_index["objective"].drop_duplicates():
    lines += [f"## {obj}", ""]
    for _, r in artifact_index[artifact_index.objective == obj].sort_values(["kind", "filename"]).iterrows():
        lines.append(f"- **{r.kind.title()}**: `{r.filename}` — {r.description}")
    lines.append("")

lines += ["## Interpretations", ""]
lines += INTERPRETATIONS

report = "\n".join(lines)
report_path = SUM / "thesis_results_output_report.md"
report_path.write_text(report, encoding="utf-8")

print(f"Done. Outputs saved to: {OUT}")
display(Markdown(report))

,kind,objective,filename,path,description
0,table,Objective 1,table_O1_1_electricity_logreg.csv,/content/drive/MyDrive/AM's/Self Development/S...,Strategy comparison table.
1,figure,Objective 1,figure_O1_1_electricity_logreg.png,/content/drive/MyDrive/AM's/Self Development/S...,Accuracy over time.
2,table,Objective 1,table_O1_1_electricity_xgb.csv,/content/drive/MyDrive/AM's/Self Development/S...,Strategy comparison table.
3,figure,Objective 1,figure_O1_1_electricity_xgb.png,/content/drive/MyDrive/AM's/Self Development/S...,Accuracy over time.
4,table,Objective 1,table_O1_1_electricity_gradual_concept_logreg.csv,/content/drive/MyDrive/AM's/Self Development/S...,Strategy comparison table.
5,figure,Objective 1,figure_O1_1_electricity_gradual_concept_logreg...,/content/drive/MyDrive/AM's/Self Development/S...,Accuracy over time.
6,table,Objective 1,table_O1_1_electricity_gradual_concept_xgb.csv,/content/drive/MyDrive/AM's/Self Development/S...,Strategy comparison table.
7,figure,Objective 1,figure_O1_1_electricity_gradual_concept_xgb.png,/content/drive/MyDrive/AM's/Self Development/S...,Accuracy over time.
8,table,Objective 1,table_O1_1_electricity_abrupt_concept_logreg.csv,/content/drive/MyDrive/AM's/Self Development/S...,Strategy comparison table.
9,figure,Objective 1,figure_O1_1_electricity_abrupt_concept_logreg.png,/content/drive/MyDrive/AM's/Self Development/S...,Accuracy over time.


Done. Outputs saved to: /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/thesis_results


# Thesis Results Output Report

Generated on: 2026-06-22 06:28:52
Input path: `/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results`
Output root: `/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/thesis_results`

## Objective 1

- **Figure**: `figure_O1_1_electricity_abrupt_concept_logreg.png` — Accuracy over time.
- **Figure**: `figure_O1_1_electricity_abrupt_concept_xgb.png` — Accuracy over time.
- **Figure**: `figure_O1_1_electricity_gradual_concept_logreg.png` — Accuracy over time.
- **Figure**: `figure_O1_1_electricity_gradual_concept_xgb.png` — Accuracy over time.
- **Figure**: `figure_O1_1_electricity_logreg.png` — Accuracy over time.
- **Figure**: `figure_O1_1_electricity_xgb.png` — Accuracy over time.
- **Figure**: `figure_O1_2_electricity_abrupt_concept_logreg.png` — Performance-cost bar chart.
- **Figure**: `figure_O1_2_electricity_abrupt_concept_xgb.png` — Performance-cost bar chart.
- **Figure**: `figure_O1_2_electricity_gradual_concept_logreg.png` — Performance-cost bar chart.
- **Figure**: `figure_O1_2_electricity_gradual_concept_xgb.png` — Performance-cost bar chart.
- **Figure**: `figure_O1_2_electricity_logreg.png` — Performance-cost bar chart.
- **Figure**: `figure_O1_2_electricity_xgb.png` — Performance-cost bar chart.
- **Table**: `table_O1_1_electricity_abrupt_concept_logreg.csv` — Strategy comparison table.
- **Table**: `table_O1_1_electricity_abrupt_concept_xgb.csv` — Strategy comparison table.
- **Table**: `table_O1_1_electricity_gradual_concept_logreg.csv` — Strategy comparison table.
- **Table**: `table_O1_1_electricity_gradual_concept_xgb.csv` — Strategy comparison table.
- **Table**: `table_O1_1_electricity_logreg.csv` — Strategy comparison table.
- **Table**: `table_O1_1_electricity_xgb.csv` — Strategy comparison table.
- **Table**: `table_O1_1_strategy_comparison_all.csv` — Combined Objective 1 table.

## O1

- **Summary**: `interpretation_O1.md` — Objective 1 Interpretation

## Objective 2

- **Figure**: `figure_O2_1_electricity_abrupt_concept_logreg_performance_trigger.png` — Representative-run retraining event timeline for Performance Trigger using selected best strategy exponential_decay.
- **Figure**: `figure_O2_1_electricity_abrupt_concept_logreg_psi_trigger.png` — Representative-run retraining event timeline for PSI Trigger using selected best strategy exponential_decay.
- **Figure**: `figure_O2_1_electricity_abrupt_concept_xgb_performance_trigger.png` — Representative-run retraining event timeline for Performance Trigger using selected best strategy exponential_decay.
- **Figure**: `figure_O2_1_electricity_abrupt_concept_xgb_psi_trigger.png` — Representative-run retraining event timeline for PSI Trigger using selected best strategy exponential_decay.
- **Figure**: `figure_O2_1_electricity_gradual_concept_logreg_performance_trigger.png` — Representative-run retraining event timeline for Performance Trigger using selected best strategy sliding_window.
- **Figure**: `figure_O2_1_electricity_gradual_concept_logreg_psi_trigger.png` — Representative-run retraining event timeline for PSI Trigger using selected best strategy sliding_window.
- **Figure**: `figure_O2_1_electricity_gradual_concept_xgb_performance_trigger.png` — Representative-run retraining event timeline for Performance Trigger using selected best strategy sliding_window.
- **Figure**: `figure_O2_1_electricity_gradual_concept_xgb_psi_trigger.png` — Representative-run retraining event timeline for PSI Trigger using selected best strategy sliding_window.
- **Figure**: `figure_O2_1_electricity_logreg_performance_trigger.png` — Representative-run retraining event timeline for Performance Trigger using selected best strategy exponential_decay.
- **Figure**: `figure_O2_1_electricity_logreg_psi_trigger.png` — Representative-run retraining event timeline for PSI Trigger using selected best strategy exponential_decay.
- **Figure**: `figure_O2_1_electricity_xgb_performance_trigger.png` — Representative-run retraining event timeline for Performance Trigger using selected best strategy sliding_window.
- **Figure**: `figure_O2_1_electricity_xgb_psi_trigger.png` — Representative-run retraining event timeline for PSI Trigger using selected best strategy sliding_window.
- **Figure**: `figure_O2_2_electricity_abrupt_concept_logreg.png` — Trigger reliability comparison after isolating memory strategy using the selected best selective strategy.
- **Figure**: `figure_O2_2_electricity_abrupt_concept_xgb.png` — Trigger reliability comparison after isolating memory strategy using the selected best selective strategy.
- **Figure**: `figure_O2_2_electricity_gradual_concept_logreg.png` — Trigger reliability comparison after isolating memory strategy using the selected best selective strategy.
- **Figure**: `figure_O2_2_electricity_gradual_concept_xgb.png` — Trigger reliability comparison after isolating memory strategy using the selected best selective strategy.
- **Figure**: `figure_O2_2_electricity_logreg.png` — Trigger reliability comparison after isolating memory strategy using the selected best selective strategy.
- **Figure**: `figure_O2_2_electricity_xgb.png` — Trigger reliability comparison after isolating memory strategy using the selected best selective strategy.
- **Figure**: `figure_O2_3_electricity_abrupt_concept_logreg.png` — Representative PSI-trigger run showing whether PSI spikes align with predictive degradation.
- **Figure**: `figure_O2_3_electricity_abrupt_concept_xgb.png` — Representative PSI-trigger run showing whether PSI spikes align with predictive degradation.
- **Figure**: `figure_O2_3_electricity_gradual_concept_logreg.png` — Representative PSI-trigger run showing whether PSI spikes align with predictive degradation.
- **Figure**: `figure_O2_3_electricity_gradual_concept_xgb.png` — Representative PSI-trigger run showing whether PSI spikes align with predictive degradation.
- **Figure**: `figure_O2_3_electricity_logreg.png` — Representative PSI-trigger run showing whether PSI spikes align with predictive degradation.
- **Figure**: `figure_O2_3_electricity_xgb.png` — Representative PSI-trigger run showing whether PSI spikes align with predictive degradation.
- **Table**: `table_O2_1_trigger_comparison.csv` — Trigger mechanism comparison using only the selected best selective strategy per dataset-model pair.
- **Table**: `table_O2_selected_best_strategy_per_dataset_model.csv` — Best selective strategy selected per dataset-model pair for trigger-isolated Objective 2 analysis.

## O2

- **Summary**: `interpretation_O2.md` — Objective 2 Interpretation

## Objective 3

- **Figure**: `figure_O3_1_combined_overall.png` — Combined tradeoff scatter plot.
- **Figure**: `figure_O3_1_electricity.png` — Dataset tradeoff scatter plot.
- **Figure**: `figure_O3_1_electricity_abrupt_concept.png` — Dataset tradeoff scatter plot.
- **Figure**: `figure_O3_1_electricity_gradual_concept.png` — Dataset tradeoff scatter plot.
- **Figure**: `figure_O3_2_electricity_abrupt_concept_logreg.png` — Memory robustness bar chart.
- **Figure**: `figure_O3_2_electricity_abrupt_concept_xgb.png` — Memory robustness bar chart.
- **Figure**: `figure_O3_2_electricity_gradual_concept_logreg.png` — Memory robustness bar chart.
- **Figure**: `figure_O3_2_electricity_gradual_concept_xgb.png` — Memory robustness bar chart.
- **Figure**: `figure_O3_2_electricity_logreg.png` — Memory robustness bar chart.
- **Figure**: `figure_O3_2_electricity_xgb.png` — Memory robustness bar chart.
- **Table**: `table_O3_1_memory_strategy_comparison.csv` — Memory strategy comparison table.

## O3

- **Summary**: `interpretation_O3.md` — Objective 3 Interpretation

## Final Summary

- **Table**: `table_final_overall_strategy_strength_weakness_summary.csv` — Overall strategy strength and weakness summary.

## Interpretations

## Objective 1 Interpretation

- Mean selective retraining accuracy was 0.8636.
- Static no-retraining accuracy was 0.8260.
- The figures emphasize robustness improvement versus retraining overhead without combining unrelated objectives.
## Objective 2 Interpretation

- After controlling for memory mechanism, performance-trigger retraining achieved mean accuracy of 0.8706, compared with 0.8678 for PSI-trigger retraining.
- Performance-trigger retraining averaged 23.33 retraining events, while PSI-trigger retraining averaged 38.33.
- Mean retraining-data cost was 209416.67 for performance-trigger retraining and 715750.00 for PSI-trigger retraining.
- Average recovery time was 1.15 for performance-trigger retraining and 1.76 for PSI-trigger retraining.
- Because the comparison uses only the selected best selective strategy per dataset-model pair, the analysis better isolates the effect of trigger mechanism from the effect of memory strategy.
## Objective 3 Interpretation

- Best mean accuracy memory strategy: Exponential Decay.
- Lowest mean retraining-data cost strategy: Sliding Window.
- Most stable strategy by lowest accuracy variability: Exponential Decay.